# 데이터 결합 — `concat` · `merge`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------
# 한글 폰트 설정 (OS 자동 감지)
# ---------------------------------------
import platform
_os = platform.system()
if _os == 'Windows':
    plt.rc('font', family='Malgun Gothic')     # 윈도우
elif _os == 'Darwin':
    plt.rc('font', family='AppleGothic')        # 맥
else:
    plt.rc('font', family='NanumGothic')        # 리눅스 / 구글 코랩
plt.rc('axes', unicode_minus=False)

---
## 1. `concat` — 이어 붙이기
같은 구조의 데이터를 위아래(행)로 잇습니다. `ignore_index=True`로 인덱스를 새로 매깁니다.

In [2]:
jan = pd.DataFrame({'월': '1월', '상품': ['아메리카노', '라떼'], '매출': [120, 95]})
feb = pd.DataFrame({'월': '2월', '상품': ['아메리카노', '라떼'], '매출': [135, 110]})
display(jan)
display(feb)

,월,상품,매출
0,1월,아메리카노,120
1,1월,라떼,95


,월,상품,매출
0,2월,아메리카노,135
1,2월,라떼,110


In [3]:
pd.concat([jan, feb], ignore_index=True)

,월,상품,매출
0,1월,아메리카노,120
1,1월,라떼,95
2,2월,아메리카노,135
3,2월,라떼,110


> 💡 `ignore_index`는 인덱스만 정리합니다. **업무 키(주문번호 등)의 중복은 따로 확인**해야 합니다 — 실전에서 중요.

---
## 2. `merge` — 공통 키로 연결
두 표를 공통 키의 값으로 짝지어 옆으로 붙입니다.

In [4]:
customers = pd.DataFrame({'customer_id':['C01','C02','C03','C04'], 
                          'region':['서울','부산','서울','대전']})
orders    = pd.DataFrame({'order_id':['O01','O02','O03','O04'], 
                          'customer_id':['C01','C02','C02','C05'],
                          'order_amount':[50000,32000,18000,41000]})
display(customers)
display(orders)

,customer_id,region
0,C01,서울
1,C02,부산
2,C03,서울
3,C04,대전


,order_id,customer_id,order_amount
0,O01,C01,50000
1,O02,C02,32000
2,O03,C02,18000
3,O04,C05,41000


### `how` 네 가지 — 어느 쪽 키를 남길까

In [5]:
# inner(기본): 양쪽에 다 있는 C01,C02
customers.merge(orders, on='customer_id')                

,customer_id,region,order_id,order_amount
0,C01,서울,O01,50000
1,C02,부산,O02,32000
2,C02,부산,O03,18000


In [6]:
# left: 모든 고객 유지(주문없는 C03,C04 NaN)
customers.merge(orders, on='customer_id', how='left')    

,customer_id,region,order_id,order_amount
0,C01,서울,O01,50000.0
1,C02,부산,O02,32000.0
2,C02,부산,O03,18000.0
3,C03,서울,NaN,NaN
4,C04,대전,NaN,NaN


In [7]:
# right: 모든 주문 유지(고객 없는 C05 NaN)
customers.merge(orders, on='customer_id', how='right')    

,customer_id,region,order_id,order_amount
0,C01,서울,O01,50000
1,C02,부산,O02,32000
2,C02,부산,O03,18000
3,C05,NaN,O04,41000


In [8]:
# outer: 양쪽 전부
customers.merge(orders, on='customer_id', how='outer')   

,customer_id,region,order_id,order_amount
0,C01,서울,O01,50000.0
1,C02,부산,O02,32000.0
2,C02,부산,O03,18000.0
3,C03,서울,NaN,NaN
4,C04,대전,NaN,NaN
5,C05,NaN,O04,41000.0


| `how` | 남기는 키 | 쓰는 상황 |
|---|---|---|
| `inner` | 공통 키 | 양쪽 정보가 다 있는 것만 |
| `left` | 왼쪽 전부 | 기준 데이터에 정보 덧붙이기 (가장 흔함) |
| `right` | 오른쪽 전부 | 오른쪽 기준 유지 |
| `outer` | 양쪽 전부 | 누락·불일치 점검 |

### 키 지정 · 타입 함정
이름이 다르면 `left_on`/`right_on`, 여러 컬럼이면 `on=[...]`.       
그리고 **문자열 `'001'` 과 숫자 `1` 은 결합되지 않습니다** — 타입을 맞추세요.

In [9]:
# 이름이 다른 키
orders2 = orders.rename(columns={'customer_id':'cust_id'})
display(customers)
display(orders2)

,customer_id,region
0,C01,서울
1,C02,부산
2,C03,서울
3,C04,대전


,order_id,cust_id,order_amount
0,O01,C01,50000
1,O02,C02,32000
2,O03,C02,18000
3,O04,C05,41000


In [10]:
# 고객id를 기준으로 merge
customers.merge(orders2, left_on='customer_id', right_on='cust_id')

,customer_id,region,order_id,cust_id,order_amount
0,C01,서울,O01,C01,50000
1,C02,부산,O02,C02,32000
2,C02,부산,O03,C02,18000


## 3. 관계 검증

### `validate`
- 두 데이터를 결합하기 전에 데이터간의 관계를 확인합니다.

| 관계 | 뜻 | 예시 | `validate` |
|---|---|---|---|
| 1:1 | 왼쪽 하나에 오른쪽 하나 | 회원 ↔ 회원 등급 |`one_to_one`|
| 1:N | 왼쪽 하나에 오른쪽 여럿 | 회원 1명 ↔ 주문 여러 건 |`one_to_many`|
| N:1 | 왼쪽 여럿에 오른쪽 하나 | 주문 여러 건 ↔ 상품 1개 |`many_to_one`|
| N:M | 왼쪽 여럿에 오른쪽 여럿 | 보통은 없는 관계 |`many_to_many`|


`validate`에 관계를 적어두면, 실제 데이터가 그와 **다를 때 에러로 알려줍니다.**

---

### `indicator` — 매칭상태를 표시해주는 도구

`indicator=True`를 주면 결과에 `_merge` 열이 생겨, 각 행이 어디서 왔는지 알려줍니다.

- `both` : 양쪽 표에 모두 있음
- `left_only` : 왼쪽 표에만 있음
- `right_only` : 오른쪽 표에만 있음

주문엔 있는데 회원 명단엔 없는 회원처럼 **짝을 못 찾은 행(미매칭)** 을 골라낼 때 씁니다. 

In [11]:
display(customers)
display(orders)
m = customers.merge(orders, how='outer', validate='one_to_many', indicator=True)

,customer_id,region
0,C01,서울
1,C02,부산
2,C03,서울
3,C04,대전


,order_id,customer_id,order_amount
0,O01,C01,50000
1,O02,C02,32000
2,O03,C02,18000
3,O04,C05,41000


`indicator=True`가 만든 `_merge` 열로 **어느 쪽에만 있는지**(`both`/`left_only`/`right_only`)를 진단합니다.

In [12]:
m['_merge'].value_counts().to_dict()

{'both': 3, 'left_only': 2, 'right_only': 1}

---
## 4. 행 단위(grain) 맞추기
**한 행의 단위(grain)** 가 다르면 
=>  한쪽을 `groupby`로 집계해 단위를 맞춘 뒤 붙입니다.

In [13]:
# 주문: 한 행 = 주문 1건
orders = pd.DataFrame({
    '주문ID': ['O1', 'O2', 'O3'],
    '고객':   ['김지은', '박민수', '이서연'],
    '주문일': ['2025-01-02', '2025-01-03', '2025-01-05'],
})

# 주문상품: 한 행 = 상품 1건  (한 주문에 여러 행이 생김)
order_items = pd.DataFrame({
    '주문ID': ['O1', 'O1', 'O2', 'O3'],
    '상품':   ['노트북', '마우스', '키보드', '모니터'],
    '금액':   [1200, 30, 50, 250],
})

display(orders)        # 그레인: 주문 단위
display(order_items)   # 그레인: 주문에 담긴 상품 1건(주문+상품 단위)


,주문ID,고객,주문일
0,O1,김지은,2025-01-02
1,O2,박민수,2025-01-03
2,O3,이서연,2025-01-05


,주문ID,상품,금액
0,O1,노트북,1200
1,O1,마우스,30
2,O2,키보드,50
3,O3,모니터,250


In [14]:
# ① 주문상품을 '주문 단위'로 집계 → 그레인을 '주문'으로 맞춤
item_agg = order_items.groupby('주문ID', as_index=False).agg(
    상품수=('상품', 'count'),
    상품합계=('금액', 'sum'),
)
item_agg   # 이제 주문ID가 고유 → 주문 1건 = 1행

,주문ID,상품수,상품합계
0,O1,2,1230
1,O2,1,50
2,O3,1,250


In [15]:
# ② 주문(1행) ↔ 집계(1행) 를 1:1 로 결합
result = orders.merge(item_agg, on='주문ID', how='left', validate='one_to_one', indicator=True)
result     # 주문 1건 = 1행, 여기에 상품수·상품합계가 붙음

,주문ID,고객,주문일,상품수,상품합계,_merge
0,O1,김지은,2025-01-02,2,1230,both
1,O2,박민수,2025-01-03,1,50,both
2,O3,이서연,2025-01-05,1,250,both


> 💡 이 '집계 후 merge'가 여러 소스를 하나로 만드는 실무의 핵심 패턴입니다. **결합 실전**에서 주문상품·주문·상품·고객을 이렇게 잇습니다.

---
# ▶ 실전 — 전자상거래 데이터 결합 (8단계)
`concat`·`merge`·`validate`·`indicator`를 실전 데이터에 적용합니다.      
이 데이터에는 **의도적으로 심어둔 오류**가 있습니다.      
문법 실행보다 **'예상 vs 실제'를 비교하며 진단**하는 데 집중하세요.

> ⚠️ **ID는 문자열!** `C0001`·`P0001`처럼 앞자리 0이 있어 `dtype=str`로 읽지 않으면 `1`로 바뀝니다.

## 1단계 · 데이터 구조 확인

이번 실전은 한 **온라인 쇼핑몰**의 데이터입니다. 성격에 따라 5개 파일로 나뉘어 있고, 우리는 이걸 결합해 하나의 분석표로 만들 겁니다.

| 파일 | 한 행의 의미 | 주요 키 | 담긴 정보 |
|---|---|---|---|
| `customers` | 고객 1명 | `customer_id` | 이름·지역·등급·가입일·연령대 |
| `orders_2025H1` / `H2` | 주문 1건 (상·하반기) | `order_id` | 고객·주문일·상태·결제수단·주문금액 |
| `order_items` | 주문에 담긴 상품 1건 | `order_item_id` | 주문·상품·수량·판매단가 |
| `products` | 상품 정보 1건 (버전 이력 포함) | `product_id` | 상품명·카테고리·브랜드·기준가 |

**연결 키**     
주문↔고객은 `customer_id`      
주문↔주문상품은 `order_id`      
주문상품↔상품은 `product_id`로 이어집니다.

> ⚠️ **ID는 문자열로 읽습니다.** `C0001`·`P0001`처럼 앞자리 0이 있어, 숫자로 읽으면 `1`이 되어 버립니다 → `dtype=str`.

In [16]:
# 데이터 불러오기
customers = pd.read_csv('data/customers.csv', dtype=str)
h1 = pd.read_csv('data/orders_2025H1.csv', dtype=str)
h2 = pd.read_csv('data/orders_2025H2.csv', dtype=str)
items = pd.read_csv('data/order_items.csv', dtype=str)
products = pd.read_csv('data/products.csv', dtype=str)

# ('데이터프레임이름', 데이터프레임)
dfs = [('고객:customers',customers), ('상반기주문:h1',h1), ('하반기주문:h2',h2), ('주문-상품:items',items), ('상품정보:products',products)]

In [17]:
for df_name, df in dfs:
  print(df_name)
  display(df.head(3))

고객:customers


,customer_id,customer_name,region,customer_grade,signup_date,age_group
0,C0001,최준서,부산,GOLD,2024-11-13,30대
1,C0002,강수빈,대전,BRONZE,2023-12-26,40대
2,C0003,조현우,부산,GOLD,2022-07-02,40대


상반기주문:h1


,order_id,customer_id,order_date,order_status,payment_method,order_total
0,O000001,C0028,2025-06-23,배송중,카드,145000
1,O000002,C0539,2025-03-10,배송완료,간편결제,110000
2,O000003,C0975,2025-03-26,배송완료,계좌이체,194000


하반기주문:h2


,order_id,customer_id,order_date,order_status,payment_method,order_total
0,O002498,C0943,2025-12-07,배송완료,무통장입금,70000
1,O002499,C0300,2025-12-19,배송완료,카드,12000
2,O002500,C0996,2025-12-29,배송완료,카드,162000


주문-상품:items


,order_item_id,order_id,product_id,quantity,unit_price
0,I0000001,O000001,P0143,1,74000
1,I0000002,O000001,P0059,1,36000
2,I0000003,O000001,P0039,1,19000


상품정보:products


,product_id,product_name,category,brand,list_price,updated_at
0,P0001,무선 마우스 001,디지털,Nexbit,46000,2024-12-18
1,P0002,텀블러 002,생활용품,HomeFit,30000,2024-12-26
2,P0003,노트 세트 003,문구,WriteOn,20000,2024-12-03


In [18]:
# 데이터프레임 정보 확인
for df_name, df in dfs:
    print("======="+df_name+"=======")
    df.info()

=======고객:customers=======
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customer_id     1000 non-null   str  
 1   customer_name   1000 non-null   str  
 2   region          1000 non-null   str  
 3   customer_grade  1000 non-null   str  
 4   signup_date     1000 non-null   str  
 5   age_group       1000 non-null   str  
dtypes: str(6)
memory usage: 47.0 KB
=======상반기주문:h1=======
<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        2500 non-null   str  
 1   customer_id     2500 non-null   str  
 2   order_date      2500 non-null   str  
 3   order_status    2500 non-null   str  
 4   payment_method  2500 non-null   str  
 5   order_total     2500 non-null   str  
dtypes: str(6)
memory usage: 117.3 K

In [19]:
# 자료형 변경
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
products['updated_at'] = pd.to_datetime(products['updated_at'])
h1['order_date'] = pd.to_datetime(h1['order_date'])
h2['order_date'] = pd.to_datetime(h2['order_date'])

h1['order_total'] = h1['order_total'].astype('int')
h2['order_total'] = h2['order_total'].astype('int')
items['quantity'] = items['quantity'].astype('int')
items['unit_price'] = items['unit_price'].astype('int')
products['list_price'] = products['list_price'].astype('int')


## 2단계 · 키 품질 점검
결합 전에 각 키의 (고유값·중복행·결측)을 표로 확인합니다.

### 잠깐 — '키(key)'가 뭔가요?
각 행을 구분하고, 다른 표와 연결할 때 짝을 맞추는 기준이 되는 컬럼입니다.
키는 두 가지 역할을 합니다.

- **구분용** — 그 표에서 한 행을 유일하게 식별 (예: `customers`의 `customer_id` = 고객 한 명, 중복 없어야 함)
- **연결용** — 다른 표를 가리켜 이어줌 (예: `order_items`의 `order_id`는 "이 상품 줄이 어느 주문인지"를 가리킴)

앞에서 `merge(on='customer_id')` 할 때 적었던 그 `customer_id`가 바로 키입니다.

그래서 결합하기 전에 **키가 믿을 만한지 — 고유한가(중복 없나), 빈 값은 없나 —** 를 먼저 확인합니다. 

In [20]:
def 키품질(df, keys):
  if isinstance(keys, str):      # 문자열이면 리스트로 (예: 'customer_id' → ['customer_id'])
        keys = [keys]
  return {"행 수":df.shape[0],
          "고유키 수":df.drop_duplicates(subset=keys).shape[0],
          "중복행 수": int(df.duplicated(subset=keys).sum()),
          "키 결측":int(df[keys].isnull().any(axis=1).sum())}

pd.DataFrame(
    {'customers의 키==>customer_id':키품질(customers, 'customer_id'),
     'h1의 키==>order_id':키품질(h1, 'order_id'),
     'h2의 키==>order_id':키품질(h2, 'order_id'),
     'items의 키==>[order_item_id, order_id]':키품질(items, ['order_item_id','order_id']),
     'products 키==>products':키품질(products, 'product_id')}
).T

,행 수,고유키 수,중복행 수,키 결측
customers의 키==>customer_id,1000,1000,0,0
h1의 키==>order_id,2500,2500,0,0
h2의 키==>order_id,2500,2500,0,0
"items의 키==>[order_item_id, order_id]",12500,12500,0,0
products 키==>products,205,200,5,0


> 🔎 `products.product_id` 의 **중복행이 0이 아님**에 주목 (5단계에서 해결).

## 3단계 · 상·하반기 연결 (`concat`) + 중복 진단

In [21]:
# 상/하반기 연결
orders = pd.concat([h1, h2], ignore_index=True)

In [22]:
# 중복 진단
print(키품질(orders, 'order_id'))

{'행 수': 5000, '고유키 수': 4997, '중복행 수': 3, '키 결측': 0}


In [23]:
# 중복 데이터 확인
orders.loc[orders['order_id'].duplicated(keep=False)].sort_values('order_id')

,order_id,customer_id,order_date,order_status,payment_method,order_total
2497,O002498,C0943,2025-06-28,결제완료,무통장입금,70000
2500,O002498,C0943,2025-12-07,배송완료,무통장입금,70000
2498,O002499,C0300,2025-06-29,결제완료,카드,12000
2501,O002499,C0300,2025-12-19,배송완료,카드,12000
2499,O002500,C0996,2025-06-30,결제완료,카드,162000
2502,O002500,C0996,2025-12-29,배송완료,카드,162000


In [24]:
# 하반기의 데이터만 남기기
orders = orders.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)

## 4단계 · 주문 ↔ 주문상품 결합 (`merge`, 1:N)

In [25]:
df = orders.merge(items, on='order_id', how='left', validate='one_to_many', indicator=True)
df.head()

,order_id,customer_id,order_date,order_status,payment_method,order_total,order_item_id,product_id,quantity,unit_price,_merge
0,O000001,C0028,2025-06-23,배송중,카드,145000,I0000001,P0143,1,74000,both
1,O000001,C0028,2025-06-23,배송중,카드,145000,I0000002,P0059,1,36000,both
2,O000001,C0028,2025-06-23,배송중,카드,145000,I0000003,P0039,1,19000,both
3,O000001,C0028,2025-06-23,배송중,카드,145000,I0000004,P0044,2,8000,both
4,O000002,C0539,2025-03-10,배송완료,간편결제,110000,I0000005,P0037,1,38000,both


In [26]:
print(df['_merge'].value_counts().to_dict())

{'both': 12500, 'left_only': 0, 'right_only': 0}


In [27]:
df = df.drop(columns='_merge')

## 5단계 · 상품정보 결합
- 상품 이력 중복 정리
  - 같은 product_id 의 여러 버전 중 '최신'만 남기기 
  - products 에는 같은 상품이 수정 이력 때문에 여러 줄 있음

In [28]:
# 중복행 확인 => 같은 상품이 리뉴얼 되어 중복 저장되어 있는 것으로 확인
cond = products.duplicated(subset='product_id', keep=False)
products.loc[cond]

,product_id,product_name,category,brand,list_price,updated_at
9,P0010,견과 세트 010,식품,DailyFood,35000,2024-12-27
49,P0050,쿠키 세트 050,식품,GoodTable,35000,2024-12-10
89,P0090,티백 세트 090,식품,DailyFood,77000,2024-11-12
129,P0130,드립백 커피 130,식품,GoodTable,43000,2024-12-26
169,P0170,그래놀라 170,식품,DailyFood,40000,2024-11-05
200,P0010,견과 세트 010 리뉴얼,식품,DailyFood,36000,2025-02-15
201,P0050,쿠키 세트 050 리뉴얼,식품,GoodTable,37000,2025-03-15
202,P0090,티백 세트 090 리뉴얼,식품,DailyFood,80000,2025-04-15
203,P0130,드립백 커피 130 리뉴얼,식품,GoodTable,47000,2025-05-15
204,P0170,그래놀라 170 리뉴얼,식품,DailyFood,45000,2025-06-15


In [29]:
# 최신 데이터만 남기기 : 최신순으로 정렬 -> 최신데이터만 남기고 중복 제거
products['updated_at'] = pd.to_datetime(products['updated_at'])
products_latest = products.sort_values('updated_at').drop_duplicates(subset='product_id', keep='last')
print(products.shape, products_latest.shape)

(205, 6) (200, 6)


In [30]:
# 주문상품 + 상품 결합
df = df.merge(products_latest, on='product_id', how='left', validate='many_to_one', indicator=True)
df.head()

,order_id,customer_id,order_date,order_status,payment_method,order_total,order_item_id,product_id,quantity,unit_price,product_name,category,brand,list_price,updated_at,_merge
0,O000001,C0028,2025-06-23,배송중,카드,145000,I0000001,P0143,1,74000,플래너 143,문구,PaperLab,74000.0,2024-12-02,both
1,O000001,C0028,2025-06-23,배송중,카드,145000,I0000002,P0059,1,36000,양말 세트 059,패션,ModeDay,36000.0,2024-12-17,both
2,O000001,C0028,2025-06-23,배송중,카드,145000,I0000003,P0039,1,19000,모자 039,패션,SimpleWear,19000.0,2024-11-09,both
3,O000001,C0028,2025-06-23,배송중,카드,145000,I0000004,P0044,2,8000,카드 지갑 044,패션,SimpleWear,8000.0,2024-11-08,both
4,O000002,C0539,2025-03-10,배송완료,간편결제,110000,I0000005,P0037,1,38000,세탁망 037,생활용품,DailyHome,38000.0,2024-12-27,both


In [31]:
# 미매칭 데이터 진단 => 미등록 상품 8건
df['_merge'].value_counts()

_merge
both          12492
left_only         8
right_only        0
Name: count, dtype: int64

In [32]:
cond = df['_merge'] != 'both'
df.loc[cond]

,order_id,customer_id,order_date,order_status,payment_method,order_total,order_item_id,product_id,quantity,unit_price,product_name,category,brand,list_price,updated_at,_merge
1299,O000533,C0282,2025-06-08,배송완료,간편결제,223000,I0001300,P9001,3,19000,NaN,NaN,NaN,NaN,NaT,left_only
2599,O001050,C0503,2025-02-17,배송완료,카드,256000,I0002600,P9002,2,44000,NaN,NaN,NaN,NaN,NaT,left_only
3899,O001578,C0390,2025-01-23,배송완료,카드,120000,I0003900,P9003,1,40000,NaN,NaN,NaN,NaN,NaT,left_only
5199,O002101,C0671,2025-03-12,배송완료,계좌이체,232000,I0005200,P9004,1,26000,NaN,NaN,NaN,NaN,NaT,left_only
6499,O002619,C0398,2025-10-11,배송완료,카드,124000,I0006500,P9005,1,37000,NaN,NaN,NaN,NaN,NaT,left_only
7799,O003135,C0490,2025-08-26,배송완료,간편결제,34000,I0007800,P9006,1,34000,NaN,NaN,NaN,NaN,NaT,left_only
9099,O003650,C0538,2025-07-24,배송완료,카드,218000,I0009100,P9007,1,24000,NaN,NaN,NaN,NaN,NaT,left_only
10399,O004152,C0756,2025-10-14,배송중,무통장입금,167000,I0010400,P9008,1,29000,NaN,NaN,NaN,NaN,NaT,left_only


In [33]:
# 진단 끝났으니 보조 열(_merge) 제거
df = df.drop(columns='_merge')

## 6단계 · 고객정보 결합

In [34]:
# 고객 정보 결합
df = df.merge(customers, on='customer_id', how='left', validate='many_to_one', indicator=True)

In [35]:
# 미매칭 데이터 확인
df.loc[df['_merge']!='both']

,order_id,customer_id,order_date,order_status,payment_method,order_total,order_item_id,product_id,quantity,unit_price,...,category,brand,list_price,updated_at,customer_name,region,customer_grade,signup_date,age_group,_merge
257,O000100,C9001,2025-05-24,배송중,간편결제,104000,I0000258,P0154,1,29000,...,패션,SimpleWear,29000.0,2024-11-16,NaN,NaN,NaN,NaT,NaN,left_only
258,O000100,C9001,2025-05-24,배송중,간편결제,104000,I0000259,P0098,3,25000,...,문구,WriteOn,25000.0,2024-12-15,NaN,NaN,NaN,NaT,NaN,left_only
2224,O000900,C9002,2025-03-05,배송완료,카드,80000,I0002225,P0135,1,30000,...,식품,FreshBox,30000.0,2024-11-30,NaN,NaN,NaN,NaT,NaN,left_only
2225,O000900,C9002,2025-03-05,배송완료,카드,80000,I0002226,P0042,1,34000,...,생활용품,DailyHome,34000.0,2024-11-06,NaN,NaN,NaN,NaT,NaN,left_only
2226,O000900,C9002,2025-03-05,배송완료,카드,80000,I0002227,P0044,2,8000,...,패션,SimpleWear,8000.0,2024-11-08,NaN,NaN,NaN,NaT,NaN,left_only
6452,O002600,C9003,2025-10-08,배송완료,계좌이체,246000,I0006453,P0167,1,57000,...,생활용품,HomeFit,57000.0,2024-11-16,NaN,NaN,NaN,NaT,NaN,left_only
6453,O002600,C9003,2025-10-08,배송완료,계좌이체,246000,I0006454,P0092,2,63000,...,생활용품,DailyHome,63000.0,2024-12-21,NaN,NaN,NaN,NaT,NaN,left_only
6454,O002600,C9003,2025-10-08,배송완료,계좌이체,246000,I0006455,P0134,1,55000,...,패션,UrbanFit,55000.0,2024-12-12,NaN,NaN,NaN,NaT,NaN,left_only
6455,O002600,C9003,2025-10-08,배송완료,계좌이체,246000,I0006456,P0153,1,8000,...,문구,DailyNote,8000.0,2024-12-12,NaN,NaN,NaN,NaT,NaN,left_only
8974,O003600,C9004,2025-09-30,결제완료,카드,67000,I0008975,P0084,1,67000,...,패션,UrbanFit,67000.0,2024-11-17,NaN,NaN,NaN,NaT,NaN,left_only


In [36]:
df = df.drop(columns='_merge')

## 7단계 · 통합 데이터 완성 & 저장
매출·월 변수를 만들고 `통합주문데이터.csv`로 저장합니다. **이 파일은 다음 파트(구조 변환)에서 이어 씁니다.**

In [37]:
df['item_sales']  = df['quantity'] * df['unit_price']
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)
df.to_csv('통합주문데이터.csv', index=False, encoding='utf-8-sig')
print('저장 완료: 통합주문데이터.csv', df.shape)

저장 완료: 통합주문데이터.csv (12500, 22)


## 8단계 · 최종 검증

결합이 다 끝났으니, **완성된 데이터가 믿을 만한지** 네 가지로 점검합니다. 하나라도 이상하면 저장·분석으로 넘어가기 전에 원인을 확인해야 합니다.

| 확인 항목 | 정상이면 | 무엇을 잡아내나 |
|---|---|---|
| 행 수 유지 | `True` | 결합으로 행이 늘지 않음(뻥튀기 없음) |
| 키 고유 | `True` | 주문상품 키가 여전히 고유 |
| 미매칭 비율 | 아주 작음 | 짝 못 찾은 데이터가 예상 범위인지 |
| 매출 교차검증 | `100%` | 주문상품 매출 합이 주문 총액과 일치 → 결합이 정확 |

특히 마지막 **교차검증**이 핵심입니다 — 한 주문의 `item_sales` 합계가 그 주문의 `order_total`과 같아야 하므로, 이게 맞으면 "주문상품이 올바른 주문에 정확히 붙었다"는 강력한 증거가 됩니다.

In [38]:
print('행수 유지:', len(df)==len(items), '/ order_item_id 고유:', df['order_item_id'].is_unique)
print('상품명 미매칭:', round(df['product_name'].isna().mean()*100,2),'% / 고객 미매칭:', round(df['customer_grade'].isna().mean()*100,2),'%')
# 교차검증 — 주문상품 매출 합계 == 주문 order_total
chk = df.groupby('order_id').agg(a=('item_sales','sum'), b=('order_total','first'))
print('order_total 일치율:', round((chk['a']==chk['b']).mean()*100,2),'%  → 결합 정확성의 증거')

행수 유지: True / order_item_id 고유: True
상품명 미매칭: 0.06 % / 고객 미매칭: 0.09 %
order_total 일치율: 100.0 %  → 결합 정확성의 증거


---
## 📋 산출물 ① 결합 검증표
| 단계 | 왼쪽 | 오른쪽 | 키 | 관계 | 결합후 행수 | 미매칭 | 처리 |
|---|---|---|---|---|---:|---:|---|
| 1 | 주문상품 | 주문(최신) | order_id | 1:N | 12,500 | 0 | 정상 |
| 2 | +상품 | 상품(최신) | product_id | N:1 | 12,500 | 8 | 미등록 목록 기록 |
| 3 | +고객 | 고객 | customer_id | N:1 | 12,500 | 11 | 미등록 목록 기록 |

## 📝 산출물 ② 결합 진단 문장 3개
> 형식: 무엇을 기준으로 결합 → 무엇을 확인 → 어떤 문제 → 어떻게 처리

1. 주문상품 기준으로 주문을 order_id로 N:1 결합했고, 행 수(12,500)와 키 고유성이 유지되어 비정상 증가는 없었다.
2. 상품 결합에서 8행이 미등록(P9001~)으로 확인돼, 삭제 대신 목록으로 남기고 기준정보 확인 대상으로 분류했다.
3. 상품 product_id가 이력으로 중복되어 N:1 검증이 실패했고, updated_at 최신 기준으로 정리해 행 뻥튀기를 막았다.